In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

In [5]:
df = pd.read_csv("../data/test2.csv")
df.head()

,num_beacons_seen,rank_1_beacon,rank_2_beacon,rank_3_beacon,gap_1_2,packets_strongest,std_rssi_across_beacons,range_rssi_across_beacons,rssi_entropy,location
0,10,2,6,5,2.000000,34,10.066041,27.5,2.295797,kitchen
1,8,2,12,10,13.500000,34,10.013984,33.5,2.072735,cafeteria
2,9,2,12,9,13.500000,34,9.326589,33.5,2.191301,cafeteria
3,10,9,6,5,8.166664,34,6.254669,22.5,2.300215,cafeteria
4,13,9,12,5,1.000000,51,8.666093,24.0,2.560139,cafeteria


In [6]:
X = df.drop(columns='location')
X.head()

,num_beacons_seen,rank_1_beacon,rank_2_beacon,rank_3_beacon,gap_1_2,packets_strongest,std_rssi_across_beacons,range_rssi_across_beacons,rssi_entropy
0,10,2,6,5,2.000000,34,10.066041,27.5,2.295797
1,8,2,12,10,13.500000,34,10.013984,33.5,2.072735
2,9,2,12,9,13.500000,34,9.326589,33.5,2.191301
3,10,9,6,5,8.166664,34,6.254669,22.5,2.300215
4,13,9,12,5,1.000000,51,8.666093,24.0,2.560139


In [7]:
y = df['location']
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

In [8]:
split_idx = int(0.7 * len(X))

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]

y_train = y_encoded[:split_idx]
y_test  = y_encoded[split_idx:]

In [ ]:
class_counts = np.bincount(y_train)
epsilon = 1e-6
class_weights = {i: max(class_counts)/(c + epsilon) for i, c in enumerate(class_counts)}
# Create a weight array for each sample
sample_weights = np.array([class_weights[y] for y in y_train])

C:\Users\laber\AppData\Local\Temp\ipykernel_8988\3630967918.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  class_weights = {i: max(class_counts)/c for i, c in enumerate(class_counts)}


In [ ]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=num_classes,
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",        # fast
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42
)

# Fit with sample weights
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

In [ ]:
y_pred = xgb_model.predict(X_test)
y_pred_prob = xgb_model.predict_proba(X_test)

# Macro F1
macro_f1 = f1_score(y_test, y_pred, average="macro")
print("Macro F1:", macro_f1)

# Top-3 accuracy
top3 = top_k_accuracy_score(y_test, y_pred_prob, k=3)
print("Top-3 Accuracy:", top3)